# Complete TODO List: Implement Gaussian Copula with Regime-Dependent Correlations

## Phase 1: Foundation - Helper Functions & Utilities

### 1.1 Create Inverse CDF Functions
**File:** portfolio_optimizer_inputs.py

- [ ] Add `_inverse_cdf()` method to handle all distribution types
  - [ ] Normal distribution inverse CDF
  - [ ] Skew-Normal distribution inverse CDF
  - [ ] Student-t distribution inverse CDF
  - [ ] Normal Inverse Gaussian (NIG) inverse CDF
  - [ ] Empirical distribution inverse CDF (quantile-based)
  - [ ] Add error handling for invalid uniform inputs (u not in [0,1])

### 1.2 Add Correlation Matrix Utilities
**File:** portfolio_optimizer_inputs.py

- [ ] Add `_make_positive_definite()` method
  - [ ] Check if matrix is positive definite
  - [ ] If not, use eigenvalue clipping or nearest PD matrix
  - [ ] Add warning when correction is applied

- [ ] Add `_validate_correlation_matrix()` method
  - [ ] Check symmetry
  - [ ] Check diagonal is all 1s
  - [ ] Check all values in [-1, 1]
  - [ ] Check positive definiteness

---

## Phase 2: Market Regime Correlation Estimation

### 2.1 Update PortfolioOptimizerInputs Class
**File:** portfolio_optimizer_inputs.py

- [ ] Add `regime_correlations` attribute to `__init__`
- [ ] Add `market_regime_asset` parameter to `__init__` (default: `'SPDR S&P 500 ETF'`)
- [ ] Add `market_regime_asset_class` parameter to `__init__` (default: `'us_equity'`)
- [ ] **IMPORTANT**: Always load S&P 500 ETF for market regime, regardless of portfolio asset class

### 2.2 Implement Market Regime Loading
**File:** portfolio_optimizer_inputs.py

- [ ] Create `_load_market_regime_labels()` method
  - [ ] **Always** load `'SPDR S&P 500 ETF'` from `'us_equity'` asset class
  - [ ] Load KAMA_MSR model for S&P 500 at `end_date`
  - [ ] Extract `regime_labels` from KAMA_MSR
  - [ ] Store as `self.market_regime_labels`
  - [ ] Add error handling if S&P 500 KAMA_MSR doesn't exist
  - [ ] Print warning if using different asset class than portfolio

### 2.3 Implement Regime Correlation Estimation
**File:** portfolio_optimizer_inputs.py

- [ ] Create `estimate_regime_correlations()` method
  - [ ] Call `_load_market_regime_labels()` to get S&P 500 regimes
  - [ ] Load returns for all portfolio assets (from their respective asset classes)
  - [ ] Align portfolio asset returns with S&P 500 regime labels by date
  - [ ] For each regime (0, 1, 2, 3):
    - [ ] Filter returns to dates where S&P 500 was in that regime
    - [ ] Compute correlation matrix across portfolio assets
    - [ ] Validate correlation matrix
    - [ ] Handle edge case: <30 observations in regime (fallback to overall correlation)
  - [ ] Store as `Dict[int, pd.DataFrame]` where key is regime ID
  - [ ] Add verbose printing option to show correlation matrices
  - [ ] Print diagnostic: "Using S&P 500 regimes to define market state for [asset_class] assets"

### 2.4 Add Cross-Asset Class Support
**File:** portfolio_optimizer_inputs.py

- [ ] Handle case where portfolio contains mix of asset classes
  - [ ] Example: `['SPDR S&P 500 ETF', 'GLD - SPDR Gold Shares', 'IEF']`
  - [ ] All assets' correlations still conditioned on S&P 500 regimes
  - [ ] Load each asset's returns from its respective data source
  - [ ] Align all to common dates with S&P 500 regime labels

### 2.5 Add Correlation Visualization
**File:** portfolio_optimizer_inputs.py

- [ ] Create `plot_regime_correlations()` method
  - [ ] Use seaborn heatmap for each regime
  - [ ] 2x2 subplot layout for 4 regimes
  - [ ] Color scale from -1 to 1
  - [ ] Annotate with correlation values
  - [ ] Title: "Correlation Matrices Conditional on S&P 500 Regime"

---

## Phase 3: Gaussian Copula Implementation

### 3.1 Update BayesianForwardSimulator
**File:** bayesian_forward_simulator.py

- [ ] Add `regime_correlations` parameter to `__init__`
- [ ] Add `asset_names` parameter to `__init__` (for multi-asset support)
- [ ] Add `market_regime_probs` parameter to `__init__` (S&P 500 forward regime probs)
- [ ] Update `forward_probs` to handle multi-asset (Dict[str, pd.DataFrame])
- [ ] Update `regime_distributions` to handle multi-asset (Dict[str, Dict[int, Dict]])

### 3.2 Implement Copula Sampling Method
**File:** bayesian_forward_simulator.py

- [ ] Create `_sample_from_copula()` method
  - [ ] Input: uniform value `u`, distribution parameters `dist_params`
  - [ ] Switch on distribution type
  - [ ] Call scipy `.ppf()` (percent point function / inverse CDF)
  - [ ] Return sampled value
  - [ ] Add error handling for edge cases (u=0, u=1)

### 3.3 Implement Multi-Asset Copula Simulation
**File:** bayesian_forward_simulator.py

- [ ] Create `simulate_multiasset_copula()` method
  - [ ] Input parameters:
    - [ ] `assets_forward_probs`: Dict[str, pd.DataFrame]
    - [ ] `assets_regime_distributions`: Dict[str, Dict[int, Dict]]
    - [ ] `regime_correlations`: Dict[int, pd.DataFrame] (based on S&P 500 regimes)
    - [ ] `market_regime_probs`: pd.DataFrame (S&P 500 forward regime probabilities)
    - [ ] `assets_transition_matrices`: Dict[str, np.ndarray]
    - [ ] `n_simulations`, `random_seed`
  - [ ] Initialize storage: Dict[str, np.ndarray] with shape (n_simulations, n_days)
  - [ ] Outer loop: for each simulation
    - [ ] Initialize regime for each asset (sample from day 0 probs)
    - [ ] Initialize S&P 500 market regime (sample from market_regime_probs day 0)
    - [ ] Inner loop: for each day
      - [ ] **Use S&P 500 regime** to select correlation matrix
      - [ ] Get correlation matrix: `corr_matrix = regime_correlations[sp500_regime]`
      - [ ] Sample correlated Gaussians: `z ~ MVN(0, Σ_regime)`
      - [ ] Transform to uniform: `u = Φ(z)` where Φ is normal CDF
      - [ ] For each asset:
        - [ ] Get asset's current regime (independent of S&P 500)
        - [ ] Get asset's regime distribution parameters
        - [ ] Sample return via `_sample_from_copula(u[i], dist_params)`
        - [ ] Store return
      - [ ] Transition S&P 500 regime for next day
      - [ ] Transition each asset's regime independently
  - [ ] Return Dict[str, np.ndarray] of simulated paths

---

## Phase 4: Integration with Portfolio Optimization

### 4.1 Update PortfolioOptimizerInputs Initialization
**File:** portfolio_optimizer_inputs.py

- [ ] In `__init__` and `quick_run()`:
  - [ ] Add `market_regime_asset` parameter (default: `'SPDR S&P 500 ETF'`)
  - [ ] Add documentation explaining S&P 500 is always used for market regime
  - [ ] Store `self.market_regime_asset = 'SPDR S&P 500 ETF'` (hardcode for now)
  - [ ] Store `self.market_regime_asset_class = 'us_equity'` (hardcode for now)

### 4.2 Update compute_portfolio_inputs() Method
**File:** portfolio_optimizer_inputs.py

- [ ] Add `method='copula'` option (alongside existing 'path_covariance')
- [ ] When `method='copula'`:
  - [ ] Call `_load_market_regime_labels()` (loads S&P 500 regimes)
  - [ ] Call `estimate_regime_correlations()` (uses S&P 500 regimes)
  - [ ] Store in `self.regime_correlations`
  - [ ] Prepare multi-asset inputs:
    - [ ] `assets_forward_probs` from all simulators
    - [ ] `assets_regime_distributions` from all simulators
    - [ ] `market_regime_probs` from S&P 500 simulator
    - [ ] `assets_transition_matrices` from all KAMA_MSR models
  - [ ] Call `simulate_multiasset_copula()` (new method to create)
  - [ ] Extract terminal returns from simulations
  - [ ] Compute μ (mean of terminal returns)
  - [ ] Compute Σ (covariance of terminal returns)
  - [ ] Store correlation matrix for diagnostics

### 4.3 Create Multi-Asset Copula Simulation Wrapper
**File:** portfolio_optimizer_inputs.py

- [ ] Create `_simulate_multiasset_copula()` method
  - [ ] Prepare inputs from `self.simulators` dictionary
  - [ ] Load S&P 500 forward regime probabilities
  - [ ] Call copula simulation function
  - [ ] Return simulated paths for all assets

### 4.4 Store Copula-Generated Simulations
**File:** portfolio_optimizer_inputs.py

- [ ] Add `copula_simulations` attribute
- [ ] Store full simulation paths (for Sortino ratio, diagnostics)
- [ ] Update `asset_simulations` to use copula simulations when method='copula'

---

## Phase 5: Validation & Diagnostics

### 5.1 Add Copula Validation Method
**File:** portfolio_optimizer_inputs.py

- [ ] Create `validate_copula_correlations()` method
  - [ ] Input: simulated returns, target regime correlations
  - [ ] Compute correlation from simulated terminal returns
  - [ ] For each regime, compare simulated vs. target correlation
  - [ ] Compute mean absolute error (MAE)
  - [ ] Print comparison tables
  - [ ] Flag if MAE > 0.10 (large discrepancy)
  - [ ] Note that validation uses S&P 500 regime definitions

### 5.2 Add Diagnostic Plots
**File:** portfolio_optimizer_inputs.py

- [ ] Create `plot_copula_diagnostics()` method
  - [ ] Subplot 1: Simulated vs. empirical correlation (scatter plot)
  - [ ] Subplot 2: Distribution of simulated returns vs. fitted distributions
  - [ ] Subplot 3: Regime correlation heatmaps (4 S&P 500 regimes)
  - [ ] Subplot 4: Time series of simulated paths (sample 100 paths)
  - [ ] Add title noting "Market regimes defined by S&P 500"

---

## Phase 6: Update Portfolio Optimizer

### 6.1 Update PortfolioOptimizer Class
**File:** portfolio_optimizer.py

- [ ] Update `from_optimizer_inputs()` to accept copula-generated inputs
- [ ] Ensure `asset_simulations` are properly passed for Sortino ratio
- [ ] Add attribute to track which method was used ('copula' vs 'path_covariance')
- [ ] Store market regime asset name if copula method used

### 6.2 Add Copula Info to Summary
**File:** portfolio_optimizer.py

- [ ] Update `summary()` method to show:
  - [ ] Method used (copula vs path covariance)
  - [ ] If copula: show "Market regimes defined by: SPDR S&P 500 ETF"
  - [ ] If copula: show regime correlation summary
  - [ ] Correlation matrix of optimized portfolio

---

## Phase 7: Testing & Validation

### 7.1 Unit Tests
**File:** `FE800_project_code/test_copula.py` (new file)

- [ ] Test `_inverse_cdf()` for all distribution types
  - [ ] Test edge cases (u=0, u=1, u=0.5)
  - [ ] Test against known quantiles
- [ ] Test `_make_positive_definite()`
  - [ ] Test with valid correlation matrix (should not change)
  - [ ] Test with invalid matrix (should fix)
- [ ] Test `_load_market_regime_labels()`
  - [ ] Test loads S&P 500 correctly
  - [ ] Test works regardless of portfolio asset class
- [ ] Test `estimate_regime_correlations()`
  - [ ] Test with us_equity assets
  - [ ] Test with universe assets (bonds, commodities)
  - [ ] Test with mixed asset classes
  - [ ] Test with insufficient data (fallback behavior)
- [ ] Test `_sample_from_copula()`
  - [ ] Test each distribution type
  - [ ] Test that samples have correct distribution

### 7.2 Integration Tests
**File:** `FE800_project_code/test_copula_integration.py` (new file)

- [ ] Test full copula workflow with us_equity assets
  - [ ] Assets: SPY, IWM (both us_equity)
  - [ ] Estimate regime correlations
  - [ ] Run copula simulation
  - [ ] Validate correlation structure
  - [ ] Check no NaN values
- [ ] Test with universe assets
  - [ ] Assets: IVV, AGG, GLD (stocks, bonds, gold)
  - [ ] Verify S&P 500 regime labels are loaded
  - [ ] Verify correlations computed across different asset classes
- [ ] Test edge case: one regime has <30 observations
- [ ] Test edge case: insufficient forward data (n_days adjustment)
- [ ] Test cross-asset class portfolio
  - [ ] Example: ['SPDR S&P 500 ETF', 'GLD - SPDR Gold Shares', 'AGG']
  - [ ] Verify all assets align to S&P 500 regimes

### 7.3 Update Validation Notebook
**File:** validate_workflow.ipynb

- [ ] Add Test 10: Copula Correlation Validation
  - [ ] Run copula method with 3-4 assets
  - [ ] Include assets from different classes if using universe
  - [ ] Validate simulated correlations match S&P 500 regime targets
  - [ ] Check MAE < 0.10 for all pairs
  - [ ] Visualize correlation comparison
  - [ ] Print "Market regime defined by: SPDR S&P 500 ETF"
- [ ] Add Test 11: Copula vs Path Covariance Comparison
  - [ ] Run both methods on same data
  - [ ] Compare optimal weights
  - [ ] Compare portfolio statistics
  - [ ] Check copula has realistic correlations
  - [ ] Verify S&P 500 regime usage in copula method

---

## Phase 8: Update Backtest Workflow

### 8.1 Update Backtest Parameters
**File:** backtest.ipynb

- [ ] Add `METHOD` parameter: 'copula' or 'path_covariance'
- [ ] **Remove** `MARKET_REGIME_ASSET` parameter (always use S&P 500)
- [ ] Add documentation cell explaining:
  - [ ] "Copula method always uses SPDR S&P 500 ETF to define market regimes"
  - [ ] "This applies regardless of portfolio asset class"
- [ ] Pass `method=METHOD` to `PortfolioOptimizerInputs.quick_run()`

### 8.2 Add Copula Diagnostics to Backtest
**File:** backtest.ipynb

- [ ] After each rebalance, save:
  - [ ] Regime correlations used (from S&P 500 regimes)
  - [ ] Copula validation metrics (MAE)
  - [ ] S&P 500 regime distribution on rebalance date
- [ ] Add summary cell showing:
  - [ ] Average correlation MAE across all rebalance dates
  - [ ] S&P 500 regime correlation stability over time
  - [ ] Note: "All correlations conditional on S&P 500 regime state"

### 8.3 Compare Copula vs Independent Simulations
**File:** `FE800_project_code/backtest_comparison.ipynb` (new file)

- [ ] Run backtest with `method='path_covariance'` (old)
- [ ] Run backtest with `method='copula'` (new)
- [ ] Compare:
  - [ ] Portfolio returns
  - [ ] Sharpe ratios
  - [ ] Turnover (weight changes)
  - [ ] Correlation structure of simulated returns
  - [ ] Realized vs. expected correlations
  - [ ] Performance across different S&P 500 regime periods
- [ ] Add analysis of regime-dependent performance

---

## Phase 9: Documentation

### 9.1 Update Docstrings
- [ ] `PortfolioOptimizerInputs._load_market_regime_labels()`
  - [ ] Document that S&P 500 is always used
- [ ] `PortfolioOptimizerInputs.estimate_regime_correlations()`
  - [ ] Document S&P 500 regime conditioning
  - [ ] Add example with cross-asset class portfolio
- [ ] `PortfolioOptimizerInputs.compute_portfolio_inputs()` 
  - [ ] Add copula method docs
  - [ ] Note S&P 500 market regime usage
- [ ] `BayesianForwardSimulator.simulate_multiasset_copula()`
  - [ ] Document market regime vs. asset regime distinction
- [ ] `_sample_from_copula()`
- [ ] `_inverse_cdf()`

### 9.2 Create Copula README
**File:** `FE800_project_code/docs/copula_methodology.md` (new file)

- [ ] Explain Gaussian copula approach
- [ ] **Highlight**: "Market regimes always defined by S&P 500"
- [ ] Describe regime-dependent correlation estimation
- [ ] Show example correlation matrices by S&P 500 regime
- [ ] Explain why S&P 500 regimes work for other asset classes
- [ ] Show example: S&P 500 HV Bear → all assets correlate more
- [ ] Explain why copula preserves marginals + correlation
- [ ] Show validation results

### 9.3 Update Main README
**File:** README.md

- [ ] Add section on copula-based simulation
-
